In [ ]:
import torch
import numpy as np
import pandas as pd
import random
import os
from collections import Counter
from sklearn import tree as sktree
from sklearn.cluster import DBSCAN
from pm4py import save_vis_petri_net
from lark import Tree, Token

from config import DATA_DIR, PROJECT_ROOT
from core import *
from utils import *
import matplotlib.pyplot as plt

# Import aggiuntivi specifici per la Sepsi (non presenti in prepare_data.ipynb)
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer

N_TRACES_GENERATED = 10000
COVERAGE = 1  # Frazione di regioni XOR/loop classificate (0.0 = tutto random, 1.0 = tutto classificato)
LOOP = True # Creiamo i classificatori per i loop?
XOR = True # Creiamo i classificatori per gli xor?
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# Caricamento del log XES della Sepsi ed esplorazione statistica
XES_PATH = PROJECT_ROOT / 'datasets' / 'Sepsis Cases - Event Log.xes'
log = xes_importer.apply(str(XES_PATH))
print(f'Tracce totali: {len(log)}')

all_activities = [e['concept:name'] for trace in log for e in trace] # Prendo tutte le attività
activity_counts = Counter(all_activities) # Conto il numero di attività
print(f'\nAttività uniche nel log: {len(activity_counts)}')
for act, cnt in sorted(activity_counts.items(), key=lambda x: -x[1]):
    print(f'{act}:{cnt}')

trace_lengths = [len(t) for t in log]
print(f'\nLunghezza tracce: min={min(trace_lengths)}, max={max(trace_lengths)}, media={sum(trace_lengths)/len(trace_lengths):.1f}')
print(f'Varianti uniche: {len(set(tuple(e["concept:name"] for e in t) for t in log))}')

In [ ]:
# Scoperta del processo con Inductive Miner (IMf)
process_tree_pm4py = pm4py.discover_process_tree_inductive(log)
print('Albero di processo scoperto (notazione pm4py):')
print(process_tree_pm4py)
print()

# Mapping attività → codice T01, T02, ...
# Le attività che iniziano con 'R' (Release, Return) colliderebbero con i nomi delle regioni
# (R0, R1, ...) in PetriNetP, che distingue regioni da task tramite il primo carattere.
activities_in_model = sorted(set(get_activities_from_tree(process_tree_pm4py))) # Tutte le attività (prese singolarmente)
activity_to_code = {act: f'T{i+1:02d}' for i, act in enumerate(activities_in_model)} # Rinomino le attività, dizionario attività: codice
code_to_activity = {code: act for act, code in activity_to_code.items()} # Dizionario codice: attività

print('Mapping attività → codice:')
for act, code in activity_to_code.items():
    print(f'{code}→{act}')
print()

sese_string = process_tree_to_sese(process_tree_pm4py, activity_to_code)
print(f'Stringa SESE generata:\n{sese_string}')

In [ ]:
# Oggetto PetriNetP e Generator
tree = PARSER.parse(sese_string)
tree = createNAryTree(tree)

complexity_nested, complexity_parallel = compute_process_complexity(tree)
print(complexity_nested)
print(complexity_parallel)

net = PetriNetP(tree)
generator = Generator(N_TRACES_GENERATED, net)
generator.generateTrace(False, no_interval=True)

In [ ]:
save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    str(PROJECT_ROOT / 'datasets' / 'sepsis_petri_net.png'),
    format="png"
)

In [ ]:
# Creazione dei decision tree per gli xor
classifier_dict_xor = {}
if XOR:
    xors_to_classify = random.sample(net.xor_regions, round(COVERAGE * len(net.xor_regions)))

    for xor in xors_to_classify:
        result = create_xor_data(xor, generator.generatedTraces)
        if result is None:
            continue

        x, y, dict_loop_step_encoding, max_len, class_to_branch = result

        dt = sktree.DecisionTreeClassifier(max_depth=5, class_weight='balanced') #Se mettiamo class_weight='balanced' è come se avessimo pompato le tracce
        dt.fit(x, y)
        classifier_dict_xor[xor] = (dt, dict_loop_step_encoding, max_len, class_to_branch)

    print(f'XOR classificati: {list(classifier_dict_xor.keys())} / {net.xor_regions}')
    print(classifier_dict_xor)

In [ ]:
# Creazione dei decision tree per 'limitare' i loop e provare a dargli un 'senso'
classifier_dict_loop = {}
if LOOP:
    loops_to_classify = random.sample(net.loop_regions, round(COVERAGE * len(net.loop_regions)))

    for loop in loops_to_classify:
        result = create_loop_data(loop, generator.generatedTraces)
        if result is None:
            continue

        x, y, dict_loop_step_encoding, max_len = result

        dt = sktree.DecisionTreeClassifier(max_depth=5, class_weight='balanced') #Se mettiamo class_weight='balanced' è come se avessimo pompato le tracce
        dt.fit(x, y)
        classifier_dict_loop[loop] = (dt, dict_loop_step_encoding, max_len)

    print(f'Loop classificati: {list(classifier_dict_loop.keys())} / {net.loop_regions}')
    print(classifier_dict_loop)

In [ ]:
for clf,_,_,ctb in classifier_dict_xor.values(): #ctb -> class to branch
    plt.figure(figsize=(10, 8))
    sktree.plot_tree(clf,
               class_names=[str(i) for i in range(len(ctb.keys()))])
    plt.show()

In [ ]:
for clf,_,_ in classifier_dict_loop.values():
    plt.figure(figsize=(10, 8))
    sktree.plot_tree(clf,
               class_names=["0", "1"])
    plt.show()

In [ ]:
import math

real_traces_decoded = []   # Lista di tracce
real_trace_times = []  # Lista dei tempi della traccia

for trace in log:
    decoded = []   # Steps della traccia corrente (in formato start/end --> il dataset ha solo start nel suo)
    times = []   # Delta in secondi per ogni step della traccia corrente
    prev_time = None

    for event in trace:
        act = event['concept:name']
        if act not in activity_to_code:  # scarto attività non presenti nel modello scoperto
            continue

        code = activity_to_code[act]
        curr_time = event['time:timestamp']

        # Delta dall'evento precedente (0 per il primo evento della traccia)
        delta = (curr_time - prev_time).total_seconds() if prev_time is not None else 0.0
        prev_time = curr_time

        # Ogni evento reale diventa DUE step: start e end.
        decoded.extend([f'start_{code}', f'end_{code}'])
        times.extend([delta, 0.01])

    real_traces_decoded.append(decoded)
    real_trace_times.append(times)

# --- Normalizzazione log1p ---
# I delta temporali hanno distribuzione heavy-tail (secondi vs giorni).
# log1p comprime i valori grandi senza perdere la struttura sui piccoli.
# Formula: t_norm = log(1 + t) / log(1 + max_t)  in [0, 1]
# Denormalizzazione all'inferenza: t_reale = exp(t_norm * max_log_delta) - 1
max_raw_delta = max(t for times in real_trace_times for t in times if t > 0)
max_log_delta = math.log1p(max_raw_delta)
real_trace_times_list = [[math.log1p(t) / max_log_delta for t in times]
                          for times in real_trace_times]

print(f'Tracce reali: {len(real_traces_decoded)}')
print(f'max_delta = {max_raw_delta:.1f}s  ({max_raw_delta/3600:.1f}h)')
print(f'max_log_delta = {max_log_delta:.4f}  (fattore denorm: exp(t_norm * questo) - 1)')


In [ ]:
# Costruisco i pattern miner temporali dalle tracce REALI.
# create_real_time_map (metodo del miner) calcola la media dei delta
# log-normalizzati reali per ogni partizione, al posto dei valori 2^i sintetici.

possible_tasks = []
for task in net.tasks:
    possible_tasks.append(f'start_{task}')
    possible_tasks.append(f'end_{task}')

k_partition = 3
max_depth    = 5
window       = 5   # deve coincidere con il window usato in assign_time (cella D)

tree_times_task_dict = {}

for task in possible_tasks:
    miner = TracePatternMiner()
    miner.root.name = task

    num_traces_for_task = 0
    for trace in real_traces_decoded:
        for idx, step in enumerate(trace):
            if step == task:
                miner.fit_trace(list(reversed(trace[:idx])), max_depth)
                num_traces_for_task += 1

    if num_traces_for_task == 0 or len(miner.nodes) == 0:
        tree_times_task_dict[task] = {
            'partitions': {'RESIDUALS': {'numTraces': 0, 'nodes': []}},
            'times_map':  {'RESIDUALS': 0.01}
        }
        continue

    partitions = miner.select_patterns(num_traces_for_task, k_partition)
    times_map  = miner.create_real_time_map(partitions, task, real_traces_decoded,
                                             real_trace_times_list, window=window)

    tree_times_task_dict[task] = {'partitions': partitions, 'times_map': times_map}
    print(task, {k: round(v, 4) for k, v in times_map.items()})


In [ ]:
# Costruisco i classifier regressivi di tempo dalle tracce REALI
all_unique_steps = set(e for trace in real_traces_decoded for e in trace)
all_unique_steps.add('PAD')
dict_task_step_encoding = {task: i for i, task in enumerate(all_unique_steps)}

classifier_dict_tasks = {}
for task in possible_tasks:
    result = create_task_data(task, real_traces_decoded, real_trace_times_list, dict_task_step_encoding)
    if result is None:
        continue
    x, y, max_len = result
    rt = sktree.DecisionTreeRegressor(max_depth=5)
    rt.fit(x, y)
    classifier_dict_tasks[task] = (rt, max_len)
    print(f'{task} | R²={rt.score(x, y):.4f}')


In [ ]:
# Rigenero le tracce utilizzando i classificatori
generator.generateTraceCond(classifier_dict_loop, classifier_dict_xor, limit=200, no_interval=True)

In [ ]:
# Creazione matrice identità delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->', '<>']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

# Codifica delle tracce generate
traceEncoded_regions, traceEncoded_tasks = get_encoding(
    generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses
)

# Trovo numero regioni e numero task effettivo
num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index   if str(i).startswith('T')])

# Creo il dataframe unico (regioni + task)
df_traces_complete = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(f'num_regions={num_regions}, num_tasks={num_tasks}, shape={df_traces_complete.shape}')

In [ ]:
'''Prendo tutte le tracce, le conto e poi creo una matrice di distanza tra tutte le possibili tracce --> mi servirà per la creazione dei cluster'''

'''all_traces = getall_traces(num_regions + num_tasks, df_traces_complete) # Ritorno la lista di tutte le tracce
trace_counts = Counter(all_traces) # Conto quante occorrenze ci sono per traccia
traces_encoded = list(trace_counts.keys()) # Prendo le tracce
trace_weights = list(trace_counts.values()) # I pesi solo le occorrenze
n_traces = len(traces_encoded) # Numero tracce diverse fra loro

print(n_traces)'''

In [ ]:
'''distance_matrix = create_distance_matrix(n_traces, traces_encoded, num_regions + num_tasks, n_workers=1) # Creo la matrice di distanza utilizzando la weighted levenshtein (con hamming_distance)
print(f'Distance matrix: {distance_matrix.shape}')'''

In [ ]:
'''# Clusterizzo
dbscan = DBSCAN(eps=25, min_samples=1, metric='precomputed')
dbscan.fit(distance_matrix, sample_weight=trace_weights)
cluster_labels = dbscan.labels_
print(f'Clusters trovati: {len(set(cluster_labels))}')

df_clusters = pd.DataFrame({
    'Trace_Type': [str(t) for t in traces_encoded],
    'Frequency': trace_weights,
    'N_Cluster': cluster_labels
})
df_clusters = df_clusters.sort_values(by=['N_Cluster', 'Frequency'], ascending=[True, False])

df_clusters'''

In [ ]:
'''Prendo tot tracce per cluster --> per bilanciare le tracce generate e per non avere prevalentemente tracce dello stesso cluster'''

'''num_trace_per_cluster = 1000
balanced_traces, balanced_columns = get_balance_traces_by_cluster(
    traces_encoded, cluster_labels, num_trace_per_cluster
)

df_traces_balanced = pd.DataFrame(balanced_columns).T
df_traces_balanced.index = df_traces_complete.index
print(df_traces_balanced)'''

In [ ]:
df_traces_balanced = df_traces_complete.copy()

# Prendo i rispettivi dataframe per le regioni e per le task
df_regions = df_traces_balanced.head(num_regions).copy()
df_tasks   = df_traces_balanced.tail(num_tasks).copy()

df_traces_balanced_T = df_traces_balanced.T.copy()

# Vocabolario completo (regioni + task)
unique_cols_complete = df_traces_balanced_T.drop_duplicates()
unique_tup_complete  = [tuple(x) for x in unique_cols_complete.values]
bit_to_id_complete   = {v: i for i, v in enumerate(unique_tup_complete)}
id_to_bit_complete   = {i: v for i, v in enumerate(unique_tup_complete)}
vocab_size_complete  = len(unique_cols_complete)
encode_complete      = lambda a: [bit_to_id_complete[tuple(x)] for x in a]
decode_complete      = lambda b: [id_to_bit_complete[x] for x in b]

# Vocabolario regioni
df_regions_T        = df_regions.T
unique_cols_regions = df_regions_T.drop_duplicates()
unique_tup_regions  = [tuple(x) for x in unique_cols_regions.values]
bit_to_id_regions   = {v: i for i, v in enumerate(unique_tup_regions)}
id_to_bit_regions   = {i: v for i, v in enumerate(unique_tup_regions)}
vocab_size_regions  = len(unique_cols_regions)
encode_regions      = lambda a: [bit_to_id_regions[tuple(x)] for x in a]
decode_regions      = lambda b: [id_to_bit_regions[x] for x in b]

# Vocabolario task
df_tasks_T        = df_tasks.T
unique_cols_tasks = df_tasks_T.drop_duplicates()
unique_tup_tasks  = [tuple(x) for x in unique_cols_tasks.values]
bit_to_id_tasks   = {v: i for i, v in enumerate(unique_tup_tasks)}
id_to_bit_tasks   = {i: v for i, v in enumerate(unique_tup_tasks)}
vocab_size_tasks  = len(unique_cols_tasks)
encode_tasks      = lambda a: [bit_to_id_tasks[tuple(x)] for x in a]
decode_tasks      = lambda b: [id_to_bit_tasks[x] for x in b]

print(f'vocab complete={vocab_size_complete}, regions={vocab_size_regions}, tasks={vocab_size_tasks}')

In [ ]:
# Assegno i tempi alle tracce generate usando i pattern dalle tracce reali
traces_balanced_decoded = [
    [step for step in trace if not step.startswith('end_L')]
    for trace in generator.generatedTraces
]

times = []
trace_times_list = []
window = 5

for trace in traces_balanced_decoded:
    this_trace_times = []
    for i, element in enumerate(trace):
        if i == 0:
            this_trace_times.append(0.0)
            times.append(0.0)
            continue
        times_map = tree_times_task_dict[element]['times_map']
        generated_time = assign_time(trace[:i], times_map, window)
        this_trace_times.append(generated_time)
        times.append(generated_time)
    trace_times_list.append(this_trace_times)

print(f'Tracce generate: {len(traces_balanced_decoded)}')
print(f'Steps totali: {len(times)}  |  max time normalizzato: {max(times):.4f}')


In [ ]:
data_complete = torch.tensor(encode_complete(df_traces_balanced_T.values), dtype=torch.long)
data_regions  = torch.tensor(encode_regions(df_regions_T.values),          dtype=torch.long)
data_tasks    = torch.tensor(encode_tasks(df_tasks_T.values),               dtype=torch.long)
data_times    = torch.tensor(times,                                          dtype=torch.float)

n = int(0.8 * len(df_traces_balanced_T))
print(f'Train={n}, Val={len(df_traces_balanced_T)-n}')
print(f'data_complete={data_complete.shape}, data_regions={data_regions.shape}, data_tasks={data_tasks.shape}, data_times={data_times.shape}')

In [ ]:
torch.save({
    # Tensori per il training
    'data_complete':      data_complete,
    'data_regions':       data_regions,
    'data_tasks':         data_tasks,
    'data_times':         data_times,
    'n':                  n,
    # Dimensioni vocabolari
    'vocab_size_complete': vocab_size_complete,
    'vocab_size_regions':  vocab_size_regions,
    'vocab_size_tasks':    vocab_size_tasks,
    'num_regions':         num_regions,
    'num_tasks':           num_tasks,
    # Dizionari per encode/decode (lambdas non picklabili, si ricostruiscono da questi)
    'bit_to_id_complete': bit_to_id_complete,
    'id_to_bit_complete': id_to_bit_complete,
    'bit_to_id_regions':  bit_to_id_regions,
    'id_to_bit_regions':  id_to_bit_regions,
    'bit_to_id_tasks':    bit_to_id_tasks,
    'id_to_bit_tasks':    id_to_bit_tasks,
    # Metadati processo
    "net":                net,
    'regions':            net.regions,
    'tasks':              net.tasks,
    # Classifiers
    'classifier_dict_tasks':   classifier_dict_tasks,
    'dict_task_step_encoding': dict_task_step_encoding,
    'max_log_delta':           max_log_delta,   # log1p(max_raw_delta) -- fattore denormalizzazione
    'max_raw_delta':           max_raw_delta,   # delta massimo in secondi (per riferimento)
    # [Sepsis] Mapping codice <-> attivita reale
    'activity_to_code':   activity_to_code,
    'code_to_activity':   code_to_activity,
}, DATA_DIR / 'prepared_data_sepsis.pt')

print(f'Salvato in {DATA_DIR / "prepared_data_sepsis.pt"}')
